# 10.11 · 自监督学习 / Self-Supervised Learning

> **课程定位 / Where this fits**
> 第 11 课，**Part 10 · 计算机视觉**（本部分收尾）。
> Lesson 11, **Part 10 · Computer Vision** (finale of this part).
>
> 监督学习要海量**人工标签**，又贵又慢。但世界上有**几乎无限的无标注图片**。**自监督学习(self-supervised learning, SSL)** 的革命性思路：**不用任何人工标签**，而是从数据本身**设计"假任务"**让模型学习，从而学到强大的通用特征。它是现代**基础模型(foundation model)** 预训练的核心(BERT、GPT、MAE 都是 SSL)。本课**从零实现 SimCLR**（对比式 SSL），在**无标签**数据上训练，再用**线性探针**证明学到的特征有多强，并用 **t-SNE** 看类别簇**自发涌现**。
> Supervised learning needs massive **human labels** — expensive and slow. Yet the world has **near-infinite unlabeled images**. **Self-supervised learning (SSL)** is the revolutionary idea: **with no human labels**, **invent "pretext tasks"** from the data itself so the model learns powerful general features. It's the core of modern **foundation model** pretraining (BERT, GPT, MAE are all SSL). We **implement SimCLR from scratch** (contrastive SSL), train on **unlabeled** data, prove the features' strength with a **linear probe**, and watch class clusters **emerge** via **t-SNE**.
>
> 💼 **实战/面试视角**："自监督 vs 监督 vs 无监督 / 对比学习 / MAE / 线性探针评估" 是表示学习/大模型岗热点。
> 💼 **Practical/interview angle:** "self- vs supervised vs unsupervised / contrastive learning / MAE / linear-probe evaluation" — representation-learning/LLM hot topics.

> 📐 **符号约定 / Notation**
> - 正样本对(positive pair) —— 同一图的两个增强视角 / two augmented views of the same image
> - 线性探针(linear probe) —— 冻结特征上只训一个线性分类器来评估特征质量 / linear classifier on frozen features

> 💡 **面试相关 / Interview-relevant**
> - "自监督学习是什么/和无监督的区别"（出镜率 ★★★★）
> - "SimCLR/对比学习的正负样本怎么构造"（★★★★★）
> - "线性探针为什么能评估表示质量"（★★★★）
> - "MAE(掩码自编码)的思想"（★★★★）

---

## 学习目标 / Learning Objectives
1. 理解自监督学习的思想与两大家族（对比式 / 掩码式）。
   Understand SSL and its two families (contrastive / masked).
2. 掌握 **SimCLR**：正负样本、NT-Xent 对比损失。
   Master SimCLR: positive/negative pairs, NT-Xent loss.
3. 在**无标签**数据上**从零训练** SimCLR。
   Train SimCLR from scratch on unlabeled data.
4. 用**线性探针**评估并证明 SSL 特征远胜随机特征。
   Use a linear probe to show SSL features >> random.
5. 用 t-SNE 观察类别簇在无标签下自发涌现。
   Watch class clusters emerge without labels via t-SNE.

## 目录 / TOC
1. [为什么自监督 ⭐](#1)
2. [SimCLR：正样本对与对比损失 ⭐](#2)
3. [无标签训练 SimCLR ⭐](#3)
4. [线性探针 + t-SNE 评估 + 小结 ⭐](#4)


<a id="1"></a>
## 1. 为什么自监督 ⭐ / Why Self-Supervised

- **监督学习**：需要 (数据, 人工标签)。标注一张医学影像可能要专家几分钟，标几百万张成本巨大。
  **Supervised:** needs (data, human labels). Labeling one medical image may take an expert minutes; millions are hugely costly.
- **无监督学习**(如聚类)：无标签，但通常**不直接学到可迁移的强特征**。
  **Unsupervised** (e.g. clustering): no labels, but usually **doesn't directly learn strong transferable features**.
- **自监督学习(SSL)**：无人工标签，但**从数据本身自动造出"监督信号"**——设计一个**假任务(pretext task)**，答案能从数据免费得到。模型为解这个假任务，被迫学到对真实任务也有用的通用特征。
  **Self-supervised (SSL):** no human labels, but **automatically creates "supervision" from the data itself** — a **pretext task** whose answer is free from the data. Solving it forces the model to learn general features useful for real tasks.

两大家族（面试点）：
Two families (interview):
- **对比式(contrastive)**：让同一图的不同"视角"互相靠近、不同图远离。代表 **SimCLR、MoCo**（本课实现 SimCLR）。
  **Contrastive:** pull different "views" of the same image together, push different images apart. E.g. **SimCLR, MoCo** (we build SimCLR).
- **掩码/生成式(masked/generative)**：遮住一部分，让模型**还原被遮的部分**。代表 **MAE**(图像)、**BERT/GPT**(文本)。
  **Masked/generative:** hide part of the input and **reconstruct it**. E.g. **MAE** (images), **BERT/GPT** (text).

**流程**：先在海量无标签数据上 SSL **预训练**学特征，再用少量标签**微调/线性探针**到下游任务（呼应 10.9/迁移学习）。
**Pipeline:** SSL **pretrain** on massive unlabeled data, then **fine-tune/linear-probe** to downstream tasks with few labels (echoing 10.9/transfer learning).


<a id="2"></a>
## 2. SimCLR：正样本对与对比损失 ⭐ / SimCLR: Positive Pairs & Contrastive Loss

SimCLR 的假任务极其巧妙：**同一张图做两次不同的随机增强，得到两个"视角"，它们应该被认为是"同一个东西"(正样本对)；而 batch 里其它图的视角都是"负样本"。**
SimCLR's pretext task is elegant: **augment one image twice differently to get two "views" — they should be recognized as "the same thing" (a positive pair); all other images' views in the batch are "negatives."**

模型(编码器 + 一个投影头)要学会：**把正样本对的表示拉近、把负样本推远**。直觉：要做到这点，编码器必须抓住图像的**本质内容**（一只猫无论怎么裁剪/翻转/调色都是猫），而忽略增强带来的表面变化——这正是我们想要的好特征。
The model (encoder + a projection head) learns to **pull the positive pair's representations together and push negatives apart**. Intuition: to do this, the encoder must capture the image's **essential content** (a cat is a cat under any crop/flip/color change), ignoring superficial augmentation differences — exactly the good features we want.

损失叫 **NT-Xent**(归一化温度交叉熵)：对每个视角，在 batch 所有其它视角里，让它的"正搭档"相似度最高（其实就是一个 2N 类的分类问题）。
The loss is **NT-Xent** (normalized temperature cross-entropy): for each view, among all other views in the batch, make its "positive partner" the most similar (essentially a 2N-way classification).

先可视化"正样本对"——同一张图的两个增强视角。
First, visualize positive pairs — two augmented views of one image.


In [ ]:
import os, numpy as np, matplotlib.pyplot as plt, seaborn as sns
import torch, torch.nn as nn, torch.nn.functional as F, torchvision
from torchvision import transforms
from torch.utils.data import DataLoader, TensorDataset
sns.set_theme(style="white")

DATA_ROOT = os.path.expanduser("~/.cache/dsfs_cv")
raw = torchvision.datasets.FashionMNIST(DATA_ROOT, train=True, download=True)
test = torchvision.datasets.FashionMNIST(DATA_ROOT, train=False, download=True, transform=transforms.ToTensor())
classes = raw.classes
imgs = (raw.data.float()/255.0).unsqueeze(1)[:8000]          # 无标签预训练图(忽略标签!) / unlabeled images (ignore labels!)

# SimCLR 的增强: 随机裁剪缩放 + 翻转 + 模糊 / SimCLR augmentations
aug = transforms.Compose([transforms.RandomResizedCrop(28, scale=(0.4,1.0)),
                          transforms.RandomHorizontalFlip(),
                          transforms.GaussianBlur(3, sigma=(0.1,1.5))])
def two_views(x): return aug(x), aug(x)                      # 同一批图的两个随机视角 / two random views

# 可视化 5 张图的正样本对 / visualize positive pairs for 5 images
demo = imgs[10:15]; v1, v2 = two_views(demo)
fig, axes = plt.subplots(2, 5, figsize=(11, 4.4))
for j in range(5):
    axes[0,j].imshow(v1[j,0], cmap="gray"); axes[0,j].axis("off")
    axes[1,j].imshow(v2[j,0], cmap="gray"); axes[1,j].axis("off")
axes[0,0].set_ylabel("视角1", fontsize=11); axes[1,0].set_ylabel("视角2", fontsize=11)
fig.suptitle("正样本对: 同一张图的两个随机增强视角(应被视为'同一个东西')"); plt.tight_layout(); plt.show()
print("正样本对 = 同一图的两个增强视角; 负样本 = batch 里其它所有图的视角")
print("SimCLR 目标: 正样本对拉近, 负样本推远 → 编码器学会抓本质内容(忽略增强带来的表面变化)")


<a id="3"></a>
## 3. 无标签训练 SimCLR ⭐ / Training SimCLR Without Labels

搭建 **编码器(encoder)** + **投影头(projection head)**，用 **NT-Xent** 损失在**无标签**图片上训练。注意：**整个训练过程不使用任何类别标签**。
Build an **encoder** + **projection head**, train with **NT-Xent** loss on **unlabeled** images. Note: **no class labels are used anywhere in training**.

(投影头是 SimCLR 的小技巧：对比损失作用在投影头的输出上，但下游任务用**编码器**的输出——投影头只在预训练时辅助。)
(The projection head is a SimCLR trick: the contrastive loss acts on the projection's output, but downstream tasks use the **encoder's** output — the projection head only helps during pretraining.)


In [ ]:
class Encoder(nn.Module):                                   # 编码器: 图 → 特征向量 / encoder: image → feature
    def __init__(s, d=64):
        super().__init__()
        s.f = nn.Sequential(nn.Conv2d(1,32,3,padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
                            nn.Conv2d(32,64,3,padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
                            nn.Flatten(), nn.Linear(64*7*7, d))
    def forward(s, x): return s.f(x)
class Projection(nn.Module):                                 # 投影头: 只在预训练用 / projection head (pretrain only)
    def __init__(s, d=64, p=32):
        super().__init__(); s.m = nn.Sequential(nn.Linear(d,d), nn.ReLU(), nn.Linear(d,p))
    def forward(s, x): return s.m(x)

def nt_xent(z1, z2, temp=0.5):
    """NT-Xent 对比损失 / normalized temperature-scaled cross entropy. z1,z2:(B,p)."""
    B = z1.shape[0]
    z = F.normalize(torch.cat([z1, z2], dim=0), dim=1)      # 2B 个归一化向量 / 2B normalized vectors
    sim = z @ z.T / temp                                    # 两两相似度 (2B×2B) / pairwise similarity
    sim.fill_diagonal_(-9e15)                               # 排除自己和自己 / mask self-similarity
    targets = torch.cat([torch.arange(B)+B, torch.arange(B)])  # 每个视角的正搭档索引 / positive partner index
    return F.cross_entropy(sim, targets)                    # 让正搭档相似度最高 / make the positive the most similar

def linear_probe(enc, n_lab=200):
    """冻结编码器, 只训一个线性分类器, 评估特征质量 / freeze encoder, train a linear classifier.
       用很少的标签(n_lab=200): SSL 价值在低标签时最明显 / few labels: SSL shines when labels are scarce."""
    enc.eval()
    with torch.no_grad(): Xtr = enc(imgs[:n_lab])           # 用冻结特征 / frozen features
    ytr = raw.targets[:n_lab]
    te_x = torch.stack([test[i][0] for i in range(2000)]); te_y = torch.tensor([test[i][1] for i in range(2000)])
    with torch.no_grad(): Xte = enc(te_x)
    clf = nn.Linear(Xtr.shape[1], 10); o = torch.optim.Adam(clf.parameters(), 1e-2); ce = nn.CrossEntropyLoss()
    dl = DataLoader(TensorDataset(Xtr, ytr), batch_size=128, shuffle=True)
    for _ in range(30):
        for xb, yb in dl: o.zero_grad(); ce(clf(xb), yb).backward(); o.step()
    return (clf(Xte).argmax(1) == te_y).float().mean().item()

torch.manual_seed(0); enc = Encoder(); proj = Projection()
acc_random = linear_probe(enc)                              # SSL 之前: 随机编码器的特征 / before SSL: random features
opt = torch.optim.Adam(list(enc.parameters())+list(proj.parameters()), 1e-3)
loader = DataLoader(TensorDataset(imgs), batch_size=512, shuffle=True)
loss_hist = []
for ep in range(30):
    enc.train(); proj.train(); ep_loss=[]
    for (xb,) in loader:
        v1, v2 = two_views(xb)                              # 两个增强视角 / two views (no labels!)
        opt.zero_grad(); loss = nt_xent(proj(enc(v1)), proj(enc(v2))); loss.backward(); opt.step()
        ep_loss.append(loss.item())
    loss_hist.append(np.mean(ep_loss))
acc_ssl = linear_probe(enc)                                # SSL 之后 / after SSL
fig, ax = plt.subplots(figsize=(7,3.6)); ax.plot(loss_hist, "o-")
ax.set_xlabel("epoch"); ax.set_ylabel("NT-Xent 对比损失"); ax.set_title("SimCLR 无标签预训练: 对比损失下降")
plt.tight_layout(); plt.show()
print(f"SimCLR 预训练完成(全程无标签), 对比损失 {loss_hist[0]:.2f} → {loss_hist[-1]:.2f}")


<a id="4"></a>
## 4. 线性探针 + t-SNE 评估 + 小结 ⭐ / Linear Probe + t-SNE Evaluation

怎么证明 SSL 学到了好特征？**线性探针(linear probe)**：**冻结**编码器，只在它的特征上训练**一个线性分类器**。如果一个简单的线性分类器就能分得好，说明特征本身已经把类别**线性可分**地组织好了——这是评估表示质量的标准方法。
How to prove SSL learned good features? **Linear probe:** **freeze** the encoder and train only **a linear classifier** on its features. If even a linear classifier does well, the features already organize classes **linearly separably** — the standard way to assess representation quality.

对比**随机(未训练)编码器** vs **SimCLR 预训练编码器**的线性探针精度。这里**只用 200 个标签**训练线性分类器——**SSL 的价值在标签稀缺时最明显**（标签越多，随机特征也能被线性层榨出不少，差距会缩小）。
Compare linear-probe accuracy of a **random (untrained)** encoder vs the **SimCLR-pretrained** encoder, using **only 200 labels** for the classifier — **SSL's value is largest when labels are scarce** (with many labels, even random features can be exploited and the gap shrinks).


In [ ]:
fig, ax = plt.subplots(figsize=(5.5,4))
bars = ax.bar(["随机编码器\n(未训练)", "SimCLR\n(无标签预训练)"], [acc_random, acc_ssl], color=["#bbb","#39c"])
for b,a in zip(bars,[acc_random,acc_ssl]): ax.text(b.get_x()+b.get_width()/2, a+0.01, f"{a:.3f}", ha="center")
ax.set_ylabel("线性探针 test 准确率 (仅200标签)"); ax.set_ylim(0,1); ax.set_title("仅200个标签时: SimCLR 特征 >> 随机特征(预训练全程没用标签!)")
plt.tight_layout(); plt.show()
print(f"线性探针(仅200标签): 随机编码器 = {acc_random:.3f}, SimCLR 编码器 = {acc_ssl:.3f}")
print(f"提升 +{acc_ssl-acc_random:.3f}: SimCLR 仅靠'对比同一图的两个视角'(无任何标签)就学到了线性可分的好特征")
print("注: SSL 价值在'少标签'时最大; 标签很多时随机特征也能被榨出不少, 差距会缩小(诚实的取舍)")


In [ ]:
# t-SNE: SimCLR 特征是否按类别聚成簇?(用标签只为上色, 训练从没用过标签) / clusters emerge without labels
from sklearn.manifold import TSNE
enc.eval()
sample_x = torch.stack([test[i][0] for i in range(1500)]); sample_y = np.array([test[i][1] for i in range(1500)])
with torch.no_grad(): feats = enc(sample_x).numpy()
emb = TSNE(n_components=2, init="pca", random_state=0, perplexity=30).fit_transform(feats)
fig, ax = plt.subplots(figsize=(7,6))
sc = ax.scatter(emb[:,0], emb[:,1], c=sample_y, cmap="tab10", s=10, alpha=0.8)
ax.set_xticks([]); ax.set_yticks([]); ax.set_title("SimCLR 特征的 t-SNE: 类别簇自发涌现(训练从未用标签!)")
fig.colorbar(sc, label="真实类别(仅用于上色)"); plt.tight_layout(); plt.show()
print("t-SNE 显示同类样本自发聚在一起 → SimCLR 在无标签下学到了有语义的表示")
print("这正是自监督的威力: 无需人工标注, 从数据本身学到可迁移的强特征")


```
自监督 SSL: 无人工标签, 从数据自身造'假任务'(pretext)学通用特征; 是基础模型预训练的核心
两大家族: 对比式(SimCLR/MoCo: 同图视角拉近,异图推远) / 掩码式(MAE/BERT: 遮住再还原)
SimCLR: 同图两次增强=正样本对, batch其它=负样本; 编码器+投影头; NT-Xent损失
为什么学到好特征: 要让正样本对靠近, 必须抓本质内容(忽略增强的表面变化)
线性探针: 冻结编码器只训线性分类器, 评估特征质量; SimCLR特征 >> 随机特征(无标签)
流程: 海量无标签 SSL 预训练 → 少量标签微调/探针下游任务(省标注)
代表: SimCLR/MoCo/BYOL/DINO(对比/蒸馏), MAE/BEiT(掩码), CLIP(图文, 见10.10)
```

### 💡 面试速查 / Interview cheat-sheet
1. **SSL 是什么**: 无人工标签, 用数据自身的假任务学特征。
   SSL: no human labels; learn from pretext tasks built from the data itself.
2. **SimCLR**: 同图两视角=正样本, 异图=负样本; NT-Xent 对比损失。
   SimCLR: two views of one image = positive, others = negatives; NT-Xent loss.
3. **为什么有效**: 拉近正样本迫使编码器抓本质内容、忽略增强差异。
   Why it works: pulling positives close forces capturing essence, ignoring augmentation.
4. **线性探针**: 冻结特征训线性分类器, 评估表示质量。
   Linear probe: train a linear classifier on frozen features to assess quality.
5. **掩码式 SSL**: MAE/BERT 遮住一部分再还原。
   Masked SSL: MAE/BERT hide part and reconstruct.

### 🎉 Part 10 完成 / Part 10 Complete
你已走完**计算机视觉**：从图像处理、CNN、经典架构、ResNet、数据增强，到检测、分割、姿态、ViT、多模态 CLIP、自监督学习。这些覆盖了 CV 工程师面试与实战的主干。下一站 **Part 11 经典 NLP** 转向文本世界。
You've completed **Computer Vision**: from image processing, CNNs, classic architectures, ResNet, augmentation, to detection, segmentation, pose, ViT, multimodal CLIP, and self-supervised learning — the backbone of CV interviews and practice. Next, **Part 11 Classic NLP** turns to text.
